In [0]:
# ================= Config =================
CATALOG = "assesment"

SOURCE_PATH = f"/Volumes/{CATALOG}/bronze/raw_data/ewevo2022.csv"

BRONZE_TABLE = f"{CATALOG}.bronze.population_raw"
SILVER_PERSONS_TABLE = f"{CATALOG}.silver.df_persons"
SILVER_MALES_TABLE = f"{CATALOG}.silver.df_males"
SILVER_FEMALES_TABLE = f"{CATALOG}.silver.df_females"

HEADER_MARKER = "90 & over"   # value in _c2 that identifies the header row (row 4)
NOTE_ROW_COUNT = 7            # the bottom 7 rows are notes
YEARS_PER_POPULATION = 21     # 2002-2022

# ================= utils =================

# Databricks notebook source
# MAGIC %md
# MAGIC ## utils
# MAGIC Reusable functions shared by the bronze, silver and gold notebooks. Each rule is written **once**.

# COMMAND ----------

from pyspark.sql import DataFrame, Window
from pyspark.sql.functions import col, current_timestamp, last, monotonically_increasing_id, row_number

# ---------------- Bronze ----------------

def read_raw_csv(path: str) -> DataFrame:
    """Read a CSV exactly as delivered: schema inference on, comma delimiter, no header row."""
    return (
        spark.read.format("csv")
        .option("inferSchema", True)   # detect column types from the values
        .option("sep", ",")            # comma-delimited
        .option("header", False)       # row 1 is a title, so columns become _c0.._c20
        .load(path)
    )


def add_metadata_columns(df: DataFrame) -> DataFrame:
    """Add 3 audit columns: original row order, source file, load time."""
    return (
        df
        .withColumn("_source_row_id", monotonically_increasing_id())   # original row order (tables don't keep order)
        .withColumn("_source_file", col("_metadata.file_path"))        # which file the data came from
        .withColumn("_ingested_at", current_timestamp())               # when it was loaded
    )


def write_delta_table(df: DataFrame, table_name: str) -> None:
    """Save a DataFrame as a Delta table (overwrite = safe to re-run). Creates the schema if needed."""
    schema_name = table_name.rsplit(".", 1)[0]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print(f"Saved {spark.table(table_name).count()} rows to {table_name}")

# ---------------- Silver ----------------

def clean_column_name(label) -> str:
    """'90 & over' -> '90_and_over'. str() because some labels (e.g. 104) are read as numbers."""
    return str(label).strip().replace("&", "and").replace(" ", "_")


def rename_from_header_row(df: DataFrame, marker_column: str, marker_value: str, columns: list) -> DataFrame:
    """Find the header row by its content, then rename the given columns using its labels."""
    header_row = df.filter(col(marker_column) == marker_value).first()
    for old_name in columns:
        df = df.withColumnRenamed(old_name, clean_column_name(header_row[old_name]))
    return df


def drop_last_rows(df: DataFrame, n: int, order_column: str = "_source_row_id") -> DataFrame:
    """Remove the last n rows, using the original file order."""
    total_rows = df.count()
    return (
        df
        .withColumn("_row_num", row_number().over(Window.orderBy(order_column)))
        .filter(col("_row_num") <= total_rows - n)
        .drop("_row_num")
    )


def fill_down(df: DataFrame, column: str, order_column: str = "_source_row_id") -> DataFrame:
    """Give every row the last non-empty value of `column` above it, in original file order."""
    window = Window.orderBy(order_column).rowsBetween(Window.unboundedPreceding, Window.currentRow)
    return df.withColumn(column, last(column, ignorenulls=True).over(window))

# ---------------- Checks ----------------

def check_row_count(df: DataFrame, expected: int, name: str) -> None:
    """Stop with a clear message if a step produced the wrong number of rows."""
    actual = df.count()
    assert actual == expected, f"{name}: expected {expected} rows, got {actual}"
    print(f"{name}: {actual} rows OK")



# ================= Gold config =================
very_old_90_over_split = f"{CATALOG}.gold.very_old_90_over_split"   # Task 5 table name

# ================= Gold helpers =================
from pyspark.sql import Column
from pyspark.sql import functions as F   # F.round, not round: avoids replacing Python's built-in round()


def to_int(column_name: str) -> Column:
    """'350,700' -> 350700: remove the thousands comma, then cast to integer."""
    return F.regexp_replace(F.col(column_name), ",", "").cast("int")


def percentage(part_column: str, total_column: str, decimals: int = 2) -> Column:
    """part / total * 100, rounded (e.g. 269570 / 350700 -> 76.87)."""
    return F.round(F.col(part_column) / F.col(total_column) * 100, decimals)


def as_percent_text(column_name: str) -> Column:
    """76.87 -> '76.87%'. For display only; keep the numeric column for queries."""
    return F.format_string("%.2f%%", F.col(column_name)).alias(column_name)